In [21]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [22]:
from tkinter import filedialog
from tkinter import Tk
from tkinter import simpledialog
Tk().withdraw()
fname = filedialog.askopenfilename(title='Select Fictrac .Dat File', filetypes=[("dat files","*.dat"), ("csv files", "*.csv"), ("all files", "*.*")])
print(fname)
# saveName = simpledialog.askstring("Save File", "Enter name of output file: ")

2025-05-22 15:23:31.732 Python[87882:31645334] +[IMKInputSession subclass]: chose IMKInputSession_Modern


/Users/grant/Desktop/fictrac-grant/Test1.dat


In [23]:
#add column names and read the file
column_names = ['frame counter', 'cam delta rotation vector x', 'cam delta rotation vector y', 'cam delta rotation vector z', 'delta rotation error score', 
'lab delta rotation vector x', 'lab delta rotation vector y', 'lab delta rotation vector z', 'cam absolute rotation vector x', 'cam absolute rotation vector y',
'cam absolute rotation vector z', 'lab absolute rotation vector x', 'lab absolute rotation vector y', 'lab absolute rotation vector z', 'lab integrated x position',
'lab integrated y position', 'integrated animal heading', 'animal direction movement', 'animal movement speed', 'integrated forward motion', 'integrated side motion', 'timestamp',
'sequence counter', 'delta timestamp', 'alt. timestamp' ]
df = pd.read_csv(fname, names = column_names)

In [24]:
from datetime import datetime
curDate = datetime.now().strftime("%m-%d-%Y_%H-%M-%S")
day = datetime.now().strftime("%m/%d/%Y")
print(curDate)

05-22-2025_15-23-35


# need ball radius value to translate data into accurate data

In [25]:
#drop unused columns
df = df.drop(columns=['lab integrated x position', 'lab integrated y position', 
                 'integrated animal heading', 'animal direction movement', 
                 'cam absolute rotation vector x', 'cam absolute rotation vector y',
                 'cam absolute rotation vector z', 'lab absolute rotation vector x', 
                 'lab absolute rotation vector y', 'lab absolute rotation vector z',
                 'cam delta rotation vector x', 'cam delta rotation vector y',
                 'cam delta rotation vector z', 'lab delta rotation vector x',
                 'lab delta rotation vector y', 'lab delta rotation vector z',
                 ])

In [26]:
#scale all relevant columns to cm by using radius of trackball
radiuscm = 9.5
df['integrated forward motion'] = df['integrated forward motion'] * radiuscm
df['integrated side motion'] = df['integrated side motion'] * radiuscm
df['animal movement speed'] = df['animal movement speed'] * radiuscm

In [27]:
#convert timestamps to seconds from ms and makes timestamp start from beginning of recording
df['timestamp'] = df['timestamp'] - df['timestamp'].iloc[0]
df['timestamp'] /= 1000
df['delta timestamp'] /= 1000

In [28]:
#changes speed from cm/frame to cm/s
df['animal movement speed'] = df['animal movement speed']/df['delta timestamp']

In [29]:
#creates a measure of acceleration
delta_v = df['animal movement speed'].diff()
delta_t = df['timestamp'].diff()

df['acceleration'] = (delta_v / delta_t).fillna(0)

In [30]:
#measures average velocity
avgvel = df['animal movement speed'].mean()

### Dont think this is needed anymore
probably don't need integrated animal heading, animal direction movement, angular orientation, lab integrate x/y position, lab absolute rotation vector

### Same as animal movement speed so not needed
### needed to calculate angular orientation tho, dist per frame not needed

In [31]:
#calculate distance walked per frame, integrated forward motion is cumulative
df['x per frame'] = df['integrated side motion'].diff().fillna(0)
df['y per frame'] = df['integrated forward motion'].diff().fillna(0)
df['dist per frame'] = np.sqrt((df['x per frame'])**2 + (df['y per frame'])**2)


In [32]:
#compute the angle of movement from one frame to the next
df['angular orientation'] = np.degrees(np.arctan2(
    df['y per frame'],
    df['x per frame']
    ))
df['angular orientation'] = df['angular orientation'].fillna(0)
#set 0 degrees as towards the speaker/+Y
df['angular orientation'] -= 90
#wrap so angles reach [-180, 180] and switch negative angles to left and positive to right
df['angular orientation'] = ((df['angular orientation'] + 180) % 360 - 180) * -1


In [33]:
#final angular orientation
finalX = df['integrated side motion'].iloc[-1]
finalY = df['integrated forward motion'].iloc[-1]
finalAngle = np.degrees(np.arctan2(finalY, finalX))
finalAngle = (finalAngle - 90)
finalAngle = ((finalAngle + 180) % 360 - 180)* -1
print(finalAngle)

15.413470882751255


In [34]:
#calculate distance walked towards, away and net from the speaker. add the distance walked in blanks
towards = df[(df['angular orientation']<=60) & (df['angular orientation']>=-60)]['dist per frame'].sum()
away = df[(df['angular orientation'] > 60) | (df['angular orientation']< -60)]['dist per frame'].sum()
total = towards+away
print("Distance Towards Speaker: ", towards, "\nDistance Away From Speaker: ", away, "\nTotal path length: ", total)

Distance Towards Speaker:  342.8756549325939 
Distance Away From Speaker:  204.38520340137913 
Total path length:  547.260858333973


In [35]:
top25thresh = df['acceleration'].quantile(0.75)
top50thresh = df['acceleration'].quantile(0.50)
dftop25 = df[df['acceleration'] >= top25thresh]
dftop50 = df[df['acceleration'] >= top50thresh]

In [36]:

size = df[['integrated forward motion', 'integrated side motion']].abs().max().max().round()
size = size + (size*0.1)

hover_template = ("<b>Timestamp: %{customdata[0]:.2f} s</b><br>" +
                        "<b>Speed: %{customdata[1]:.2f} cm/s</b><br>" +
                        "<b>Acceleration: %{customdata[2]:.2f} cm/s²</b><br>" +
                        "<b>Angular orientation: %{customdata[3]:.2f}°</b><br>" +
                        "<extra></extra>") 

px_graph = px.line(df, x='integrated side motion', y='integrated forward motion', 
                    labels = {
                        'integrated side motion': '',
                        'integrated forward motion': ''
                    },
                    hover_data={
                       'acceleration': False,
                       'integrated side motion': False,
                       'integrated forward motion': False},
                    range_x=[-size,size],
                    range_y=[-size,size],
                   )

px_graph.update_traces(customdata=df[['timestamp', 'animal movement speed', 'acceleration', 'angular orientation']],
                        hovertemplate=hover_template,)

px_graph.add_trace(go.Scatter(
    x=dftop25['integrated side motion'],
    y=dftop25['integrated forward motion'],
    mode='markers',
    marker=dict(
        size=7,
        color='red',
        opacity=0.6,
        line=dict(width=1, color='black')
    ),
    name='Top 25% Acceleration',
    visible=False,
    customdata=dftop25[['timestamp', 'animal movement speed', 'acceleration', 'angular orientation']],
    hovertemplate=hover_template))

px_graph.add_trace(go.Scatter(
    x=dftop50['integrated side motion'],
    y=dftop50['integrated forward motion'],
    mode='markers',
    marker=dict(
        size=7,
        color='orange',
        opacity=0.6,
        line=dict(width=1, color='black')
    ),
    name='Top 50% Acceleration',
    visible=False,
    customdata=dftop50[['timestamp', 'animal movement speed', 'acceleration', 'angular orientation']],
    hovertemplate=hover_template))

px_graph.add_annotation(
    text=f"Total path length: {total:.2f} cm",
    xref="paper", yref="paper",
    x=1.04, y=0.95,  
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="lightgray",
    bordercolor="black",
    borderwidth=1,
    xanchor='left'
)

px_graph.add_annotation(
    text=f"Towards speaker: {towards:.2f} cm",
    xref="paper", yref="paper",
    x=1.04, y=.9, 
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="lightgray",
    bordercolor="black",
    borderwidth=1,
    xanchor='left'
)

px_graph.add_annotation(
    text=f"Away from speaker: {away:.2f} cm",
    xref="paper", yref="paper",
    x=1.04, y=0.85,  
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="lightgray",
    bordercolor="black",
    borderwidth=1,
    xanchor='left'
)

px_graph.add_annotation(
    text=f"Final angular orientation: {finalAngle:.2f}°",
    xref="paper", yref="paper",
    x=1.04, y=0.8,  # top-right corner
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="lightgray",
    bordercolor="black",
    borderwidth=1,
    xanchor='left'
)

px_graph.add_annotation(
    text=f"Average velocity: {avgvel:.2f} cm/s",
    xref="paper", yref="paper",
    x=1.04, y=.75,  # top-right corner
    showarrow=False,
    font=dict(size=14, color="black"),
    bgcolor="lightgray",
    bordercolor="black",
    borderwidth=1,
    xanchor='left'
)

px_graph.add_annotation(
    text=f"Trackball size: {radiuscm:.2f} cm",
    xref="paper", yref="paper",
    x=1.04, y=0,  # top-right corner
    showarrow=False,
    font=dict(size=14, color="black"),
    xanchor='left'
)

px_graph.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=1.0,
            y=1.06,
            xanchor="right",
            showactive=True,
            buttons=[
                dict(
                    label="Path only",
                    method="update",
                    args=[{"visible": [True, False, False]}]  # Only path is visible
                ),
                dict(
                    label="Top 25% Accel values",
                    method="update",
                    args=[{"visible": [True, True, False]}]  # Path + top 25% markers
                ),
                dict(
                    label="Top 50% Accel values",
                    method="update",
                    args=[{"visible": [True, False, True]}]  # Path + top 50% markers
                )
            ]
        )
    ]
)

px_graph.update_layout(autosize = True,
                        yaxis_scaleanchor = "x", 
                       margin=dict(r=350),
                       xaxis_title="x position (cm)",
                        yaxis_title="y position (cm)",
                        title = {'text': f'Fictive Path of Animal<br><sup>Date: {curDate}</sup>',
                                'x': .062,
                                'y': 0.96,  
                                'xanchor': 'left', 
                                'yanchor': 'top', 
                                'font': {'size': 22}  
                               }
                        )




px_graph.show()

In [37]:
dftop25

,frame counter,delta rotation error score,animal movement speed,integrated forward motion,integrated side motion,timestamp,sequence counter,delta timestamp,alt. timestamp,acceleration,x per frame,y per frame,dist per frame,angular orientation
2,2,5840.651777,17.824956,0.997094,0.363043,0.0634,2,0.031382,5.789980e+07,64.270734,0.231925,0.509040,0.559384,24.494601
5,5,7753.277419,14.920991,2.355535,1.101752,0.1753,5,0.032054,5.789991e+07,98.623925,0.308040,0.365869,0.478277,40.095408
12,12,8670.080938,18.761945,4.274215,2.949157,0.3997,12,0.030282,5.790014e+07,330.112515,0.416895,0.385995,0.568149,47.203983
16,16,9981.368421,13.850100,5.681010,3.825935,0.5437,16,0.032548,5.790028e+07,164.292358,0.087716,0.442178,0.450794,11.220189
20,20,11408.153025,12.376361,7.010084,4.608230,0.6723,20,0.032093,5.790041e+07,65.017856,0.150029,0.367770,0.397195,22.192661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3913,3913,9657.729522,11.377918,233.658879,66.764617,130.7518,3913,0.031996,5.803049e+07,118.064218,-0.203560,-0.301820,0.364049,-146.002613
3915,3915,9276.400171,12.192458,233.480003,66.253413,130.8163,3915,0.032808,5.803055e+07,221.099984,-0.392506,-0.077107,0.400008,-101.114119
3917,3917,8924.162543,8.507660,233.555386,65.784197,130.8796,3917,0.031718,5.803062e+07,54.226583,-0.254733,0.089034,0.269844,-70.734595
3918,3918,9993.308874,10.293522,233.757078,65.525741,130.9115,3918,0.031849,5.803065e+07,55.983319,-0.258455,0.201692,0.327840,-52.032449


### trying to make plot with go

In [38]:
fig = make_subplots(rows=5, cols=2, shared_xaxes=True, horizontal_spacing=0.05,
                    specs=[[{'rowspan': 5}, {'type': 'domain'}],[None, None],[None, None],[None, None],[None, None]],
                    subplot_titles=('Fictive Path', 'Speed', 'Angular Orientation', 'Acceleration', 'Distance per frame'),
                    column_widths=[1, 0.3])

fig.add_trace(go.Scatter(x=df['integrated side motion'], 
                         y=df['integrated forward motion'],
                         mode='lines',
                         customdata=df[['timestamp', 'animal movement speed', 'angular orientation', 'acceleration']],
                         hovertemplate='Time: %{customdata[0]:.2f} s<br>' +
                                        'Speed: %{customdata[1]:.2f} cm/s<br>' +
                                        'Angle: %{customdata[2]:.2f}°<br>' +
                                        'Acceleration: %{customdata[3]:.2f} cm/s²',
                         
                         ), row=1, col=1)

fig.add_trace(go.Indicator(mode='number',
                           value=total,
                           title={'text': 'Total path length<br><span style="font-size:0.8em;color:gray">cm</span>'},
                           domain={'x': [0.7, 1], 'y': [0.9, 1]}), row=1, col=2)

highlight = df['acceleration'] > 0.9

fig.add_trace(go.Scatter(
    x=df['integrated side motion'][highlight],
    y=df['integrated forward motion'][highlight],
    mode='markers',
    name='High Speed',
    marker=dict(color='red', size=10, symbol='diamond'),
    visible=False
))

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=list([
                dict(label="Show Highlights",
                     method="update",
                     args=[{"visible": [True, True]},
                           {"title": "High-Speed Points Highlighted"}]),
                dict(label="Hide Highlights",
                     method="update",
                     args=[{"visible": [True, False]},
                           {"title": "Original Path"}]),
            ]),
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    title="Animal Path with Highlight Toggle"
)

fig.update_layout(title_text='Fictive Path', 
                  title_x=0.5, 
                  width=1300, 
                  height=900,
                  xaxis = dict(range=[-size,size]),
                  yaxis = dict(range=[-size,size]),)
fig.show()

In [39]:
#open a dialog to save the plot then close the dialog when ok or cancel is pressed
from tkinter import simpledialog
from tkinter import Tk
# Tk().withdraw()
# save_path = simpledialog.askstring("Save Plot", "Enter the path to save the plot (leave blank to skip saving):")
# print(save_path)
# if save_path:
#     fig.write_html(f"{save_path}/Fictrac_Plot_{curDate}.html")
#     print(f"Plot saved to {save_path}/Fictrac_Plot_{curDate}.html")
# else:
#     print("Plot not saved.")


In [40]:
# px_graph.write_html("/Users/grant/fictrac/" + curDate+".HTML")